# 04 — Financial PIT Walkthrough

Tujuan: memahami **point-in-time financial data** dengan satu ticker dan satu decision timestamp.

Gunakan historical Financial PIT panel yang sudah accepted. Notebook ini read-only dan tidak membuka model outcome/performance.

In [ ]:
from pathlib import Path
import pandas as pd

FINANCIAL_PANEL = Path(r"CHANGE_ME")
TICKER = "BBCA"
DECISION_TIME_WIB = "2025-05-15 18:00:00"

In [ ]:
if not FINANCIAL_PANEL.exists():
    raise FileNotFoundError("Set FINANCIAL_PANEL ke accepted historical feature_panel.parquet lokal.")

fin = pd.read_parquet(FINANCIAL_PANEL) if FINANCIAL_PANEL.suffix.lower() == ".parquet" else pd.read_csv(FINANCIAL_PANEL)
print("shape:", fin.shape)
print("tickers:", fin["ticker"].nunique() if "ticker" in fin else "?")

## 1. Lihat semua state satu ticker

In [ ]:
cols = [c for c in [
    "ticker", "fiscal_year", "feature_id", "period_stratification_key",
    "feature_value", "availability_status", "reporting_version_id",
    "reporting_publication_at_utc", "reporting_knowledge_at_utc",
    "reporting_period_start", "reporting_period_end", "reporting_instant_date",
] if c in fin.columns]
one = fin[fin["ticker"].astype(str).str.upper().eq(TICKER)].copy()
one["reporting_knowledge_at_utc"] = pd.to_datetime(one["reporting_knowledge_at_utc"], utc=True, errors="coerce")
display(one.sort_values("reporting_knowledge_at_utc")[cols].tail(40))

## 2. Apa yang sudah diketahui market pada decision time?

Decision cutoff project: 18:00 Asia/Jakarta. Filing yang knowledge time-nya lewat cutoff tidak boleh ikut state hari itu.

In [ ]:
decision_wib = pd.Timestamp(DECISION_TIME_WIB, tz="Asia/Jakarta")
decision_utc = decision_wib.tz_convert("UTC")
known = one[one["reporting_knowledge_at_utc"].le(decision_utc)].copy()
print("decision WIB:", decision_wib)
print("decision UTC:", decision_utc)
print("known states:", len(known))

## 3. Pilih latest coherent reporting-period bundle

Urutkan berdasarkan **economic reporting chronology** dulu; knowledge time hanya memilih revision terbaru di logical period yang sama. Jangan memilih filing paling baru berdasarkan knowledge time saja.

In [ ]:
period_map = {"tw1": "Q1", "q1": "Q1", "h1": "H1", "tw2": "H1", "9m": "9M", "tw3": "9M", "fy": "FY", "audit": "FY"}
period_rank = {"Q1": 1, "H1": 2, "9M": 3, "FY": 4}

work = known.copy()
work["period"] = work["period_stratification_key"].astype(str).str.lower().map(period_map)
work["period_rank"] = work["period"].map(period_rank)
work["fiscal_year_num"] = pd.to_numeric(work["fiscal_year"], errors="coerce")
period_end = pd.to_datetime(work.get("reporting_period_end"), errors="coerce")
instant = pd.to_datetime(work.get("reporting_instant_date"), errors="coerce")
work["period_date"] = period_end.fillna(instant)

valid = work.dropna(subset=["fiscal_year_num", "period_rank", "period_date"]).copy()
if valid.empty:
    raise ValueError("No valid reporting-period chronology for this ticker/time.")

latest_key = (
    valid[["fiscal_year_num", "period_rank", "period_date"]]
    .drop_duplicates()
    .sort_values(["fiscal_year_num", "period_rank", "period_date"])
    .iloc[-1]
)
same_period = valid[
    valid["fiscal_year_num"].eq(latest_key["fiscal_year_num"])
    & valid["period_rank"].eq(latest_key["period_rank"])
    & valid["period_date"].eq(latest_key["period_date"])
]
latest_knowledge = same_period["reporting_knowledge_at_utc"].max()
bundle = same_period[same_period["reporting_knowledge_at_utc"].eq(latest_knowledge)].copy()

print("selected period:", bundle["period"].iloc[0], int(bundle["fiscal_year_num"].iloc[0]))
print("latest revision knowledge:", latest_knowledge)
display(bundle[cols].sort_values("feature_id"))

## 4. CORE3 Financial V2

In [ ]:
CORE3 = [
    "leverage_liabilities_to_assets",
    "liquidity_cash_to_assets",
    "margin_net_income_to_revenue",
]
core = bundle[bundle["feature_id"].isin(CORE3)][["feature_id", "feature_value", "availability_status"]]
display(core)
print("all CORE3 AVAILABLE:", set(core["feature_id"]) == set(CORE3) and core["availability_status"].eq("AVAILABLE").all())

**Penting:** kalau satu CORE3 feature missing di current bundle, jangan mengambil feature itu dari FY/Q1 lama. Itu akan mencampur economic states berbeda.

Setelah notebook ini kamu harus bisa menjelaskan: (1) reporting period vs knowledge time, (2) kenapa revision dipilih setelah logical period, (3) kenapa post-cutoff filing menyebabkan leakage, dan (4) kenapa cross-period fallback dilarang.